## text2grammar Experiments testing


## Task2:  

Sentence → Multiple Grammars and Multiple Labels

给定一个语法，多个句子

In [6]:
import pandas as pd 
import os
import pickle
from datetime import datetime, timezone
from openai import OpenAI  # pip install openai
import nltk
from nltk.tokenize import sent_tokenize
import os
from pprint import pprint
import pandas as pd 
from mt_reasoning.utils import prompts_util, clients_util 
from tqdm import tqdm
import importlib
from dotenv import load_dotenv
import random
import string

load_dotenv()

source_df = pd.read_json("data/extraction_pdf/datasets/df_samples.jsonl", lines=True)

## uv run vllm serve /home/snt/projects_lujun/base_models/gemma-2-2b-it --host 0.0.0.0 --port 1997 --max-model-len 2048 --max-num-seqs 2 --gpu-memory-utilization 0.7


In [7]:
importlib.reload(prompts_util)
importlib.reload(clients_util)

nltk.download('punkt')

## Open AI Settings
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0.5"))


## VllM settings
model_vllm = os.environ.get("MODEL_VLLM", "/home/snt/projects_lujun/base_models/gemma-2-2b-it")
IP = os.environ.get("VLLM_IP", "0.0.0.0")
PORT = os.environ.get("VLLM_PORT", "1997")
server_url = f"http://{IP}:{PORT}/v1"
print (server_url)
vllm_client = OpenAI(base_url=server_url)

## Experimental Settings
sentence_list_size = 3
letters = list(string.ascii_uppercase)  # ['A', 'B', 'C', ..., 'Z']

time_now = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = "/home/snt/projects_lujun/mt_reasoning/data/extraction_pdf/datasets"
output_path = os.path.join(output_dir, f"task2_{time_now}_{sentence_list_size}_{model_vllm.split('/')[-1]}.jsonl")

http://0.0.0.0:1997/v1


[nltk_data] Downloading package punkt to /home/snt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
grammar_df = source_df.drop_duplicates(subset=['grammar_points_descriptions'])

print (f"Unique grammar descriptions: {len(grammar_df)}")

for index, row in tqdm(grammar_df.iterrows(), total=len(grammar_df)):
    grammar_desc = row['grammar_points_descriptions']
    sentence_lux = row['luxembourg']
    opposite_source_sentence_list = grammar_df[grammar_df['grammar_points_descriptions'] != grammar_desc]['luxembourg'].sample(sentence_list_size-1).tolist()
    full_list = [sentence_lux] + opposite_source_sentence_list
    random.shuffle(full_list)
    sentence_index = full_list.index(sentence_lux)

    assert sentence_index != -1, "Sentence not found in the list."
    assert len(full_list) == sentence_list_size, "Sentence list size mismatch."

    option_labels = letters[:sentence_list_size]
    correct_sentence_letter = option_labels[sentence_index]
    labeled_sentence_list = [
        f"{label}. {desc}" for label, desc in zip(option_labels, full_list)
    ]
    input_dict = {
        "GRAMMAR_POINT": grammar_desc,
        "LIST_OF_SENTENCES": "\n".join(labeled_sentence_list),
    }


    output_dict, input_prompt = clients_util.generate_with_calling_api(
        client=vllm_client,
        system_prompt_template_path="prompts/system/system_prompt_translation.jinja",
        input_prompt_template_path="prompts/evaluation/prompt_sentence_classification_task_2.jinja",  # Use simple, complecated one confuse the models
        input_text_dict=input_dict,
        model=model_vllm,
    )
    
    row["input_prompt"] = input_prompt
    row["task2_dict"] = output_dict
    row["correct_sentence_letter"] = correct_sentence_letter
    updated_row = pd.DataFrame([row])
    if not os.path.exists(output_dir):  
        os.makedirs(output_dir)
    if not os.path.exists(output_path):
        updated_row.to_json(output_path, orient="records", lines=True)
    else:
        updated_row.to_json(output_path, orient="records", lines=True, mode="a")
    
    # print(output_dict)
    # print("----------------------------------------------")
    # pprint(output_dict, indent=2, width=150, sort_dicts=False)

In [10]:
result_df = pd.read_json("data/extraction_pdf/datasets/task2_20251002_143257_3_gemma-2-2b-it.jsonl", lines=True)

In [11]:
num_correct = 0
total = len(result_df)
for index, row in result_df.iterrows():
    correct_grammar_letter = row['correct_sentence_letter']
    detected_grammar_letter = row['task2_dict'].get('sentence_selected', '').strip().upper()
    if correct_grammar_letter == detected_grammar_letter:
        num_correct += 1
print(f"Accuracy: {num_correct}/{total} = {num_correct/total:.2%}")

Accuracy: 553/673 = 82.17%
